<a href="https://colab.research.google.com/github/Elwing-Chou/tibame20260427/blob/main/tibame20260606.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
上次的複習(爬蟲整個流程)

1. 找到網址: a. 原始碼有  b. 原始碼沒有(F12找隱藏)
2. 解析看格式: a. JSON格式  b. HTML格式

分支1. JSON格式
1. json.loads: 字典/list解析

分支2. HTML格式
2. BeautifulSoup: 找你需要的區塊(html名字+class做篩選)

排版屬性: 在區塊上加上這些屬性來輔助排版
a. class: class="分類1 分類2"(熟悉)
b. id: id="xxx"(補充)
html.find(名字, {"id":"xxxx"})

3. 儲存+分析(pandas)


```


In [ ]:
import urllib.request as req
import bs4 as bs

def get_page(page_num):
    url = f"https://tabelog.com/tw/tokyo/rstLst/sweets/{page_num}/?SrtT=rt"
    print(url)
    resp = req.urlopen(url)
    content = resp.read()
    html = bs.BeautifulSoup(content)
    rs = html.find_all("div", {"class":"list-rst__body"})

    # 最後回傳的一頁的餐廳清單
    result = []
    for r in rs:
        name = r.find("a", {"class":"list-rst__rst-name-target"})
        area_genre = r.find("div", {"class":"list-rst__area-genre"})
        rating = r.find("span", {"class":"c-rating__val"})
        prices = r.find_all("span", {"class":"c-rating-v3__val"})
        dinner_price = prices[0]
        lunch_price = prices[1]
        holiday = r.find("span", {"class":"list-rst__holiday-text"})
        imgs = r.find_all("img", {"class":"js-thumbnail-img"})

        name_text = name.get_text().strip()
        name_href = name["href"]
        area_genre_text = area_genre.get_text().strip()
        rating_text = rating.get_text().strip()
        dinner_price_text = dinner_price.get_text().strip()
        lunch_price_text = lunch_price.get_text().strip()
        holiday_text = holiday.get_text().strip()

        imgs_src = []
        for img in imgs:
            imgs_src.append(img["src"])

        # 一筆資料是一個字點
        data = {
            "rating":rating_text,
            "name":name_text,
            "link":name_href,
            "area_genre":area_genre_text,
            "price_dinner":dinner_price_text,
            "price_lunch":lunch_price_text,
            "holiday":holiday_text,
            "imgs":imgs_src,
        }
        result.append(data)

    return result

get_page(2)

In [9]:
# [{}餐廳, {}餐廳] -> dataframe
import pandas as pd

total = []
for i in range(5):
    page = i + 1
    partial = get_page(page)
    total = total + partial

df = pd.DataFrame(total)
df

https://tabelog.com/tw/tokyo/rstLst/sweets/1/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/2/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/3/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/4/?SrtT=rt
https://tabelog.com/tw/tokyo/rstLst/sweets/5/?SrtT=rt


,rating,name,link,area_genre,price_dinner,price_lunch,holiday,imgs
0,4.24,Bon.nu,https://tabelog.com/tw/tokyo/A1304/A130401/131...,"參宮橋車站 356m / 法式料理, 牛排, 蛋糕","JPY 50,000 - JPY 59,999","JPY 50,000 - JPY 59,999",-,"[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
1,4.20,Okashiya Ucchi,https://tabelog.com/tw/tokyo/A1309/A130901/132...,"北參道車站 250m / 蛋糕, 西式甜點",-,"JPY 5,000 - JPY 5,999",星期一,"[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
2,4.16,蒼菓,https://tabelog.com/tw/tokyo/A1307/A130703/132...,廣尾車站 702m / 甜食,"JPY 15,000 - JPY 19,999","JPY 15,000 - JPY 19,999","星期一, 星期二, 星期日","[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
3,4.15,Yama,https://tabelog.com/tw/tokyo/A1316/A131602/132...,白金台車站 725m / 甜食,"JPY 20,000 - JPY 29,999","JPY 30,000 - JPY 39,999","星期二, 星期三","[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
4,4.05,ESPRIT C. KEI GINZA,https://tabelog.com/tw/tokyo/A1301/A130101/132...,"銀座車站 330m / 法式料理, 甜食","JPY 30,000 - JPY 39,999",-,"星期一, 星期日","[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
...,...,...,...,...,...,...,...,...
95,3.77,秘密堂,https://tabelog.com/tw/tokyo/A1311/A131106/131...,"日暮里車站 376m / 刨冰, 日式甜點店","JPY 2,000 - JPY 2,999","JPY 1,000 - JPY 1,999","星期一, 星期二","[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
96,3.77,Fruit Parlour Goto,https://tabelog.com/tw/tokyo/A1311/A131102/130...,"淺草車站 224m / 水果聖代, 咖啡店, 刨冰","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999","星期三, 星期四","[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
97,3.77,MAISON LANDEMAINE 麻布台,https://tabelog.com/tw/tokyo/A1307/A130701/131...,"六本木一丁目車站 429m / 麵包, 西式甜點, 咖啡店","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999",-,"[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
98,3.77,FRENCH POUND HOUSE 大和鄉本店,https://tabelog.com/tw/tokyo/A1323/A132301/130...,"巢鴨車站 228m / 蛋糕, 咖啡店","JPY 1,000 - JPY 1,999","JPY 1,000 - JPY 1,999",-,"[data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP/..."
